# 03 · IMF selection for each noise (paper Sec. III-A, Fig. 2)

Each noise is added **alone** at SNRin = 10 dB to the 13 records of Fig. 10. The LMS filter is applied to the sum of
IMFi..IMFj (every contiguous range) or to the whole signal ("no EMD"), and the unused IMFs and residue are added back.

The paper's result:

| noise | best input to the adaptive filter |
|---|---|
| PLI | IMF1 |
| BW | IMF5–8 |
| EM | IMF3–8 |
| MA | whole signal (no EMD) |

In [ ]:
from pathlib import Path
import sys, json

project_root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

RESULTS = project_root / "results"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
METHOD_COLORS = {"PEAF": "#1f3b73", "SEAF": "#9aa83a", "PDAF": "#d62728", "SDAF": "#3cb9a0"}


In [ ]:
imf = pd.read_csv(RESULTS / "imf_selection.csv")
print("LMS setting:", json.loads((RESULTS / "lms_setting.json").read_text()))
PAPER = {"pli": "IMF1", "bw": "IMF5-8", "em": "IMF3-8", "ma": "no EMD"}
mean = imf.groupby(["noise", "combination"]).SNRimp_dB.mean()
summary = pd.DataFrame({
    n: {"paper choice": PAPER[n], "SNRimp paper choice": mean[n][PAPER[n]],
        "best here": mean[n].idxmax(), "SNRimp best": mean[n].max(), "SNRimp no EMD": mean[n]["no EMD"],
        "paper choice is best": mean[n].idxmax() == PAPER[n]}
    for n in PAPER}).T
summary

## Mean SNRimp for every IMF range (box: paper's choice; title: no-EMD value)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4.6))
for ax, n in zip(axes, PAPER):
    M = np.full((8, 8), np.nan)
    for i in range(1, 9):
        for j in range(i, 9):
            M[i - 1, j - 1] = mean[n][f"IMF{i}" if i == j else f"IMF{i}-{j}"]
    im = ax.imshow(M, cmap="viridis")
    for i in range(8):
        for j in range(i, 8):
            ax.text(j, i, f"{M[i, j]:.1f}", ha="center", va="center", fontsize=6.5,
                    color="w" if M[i, j] < np.nanmax(M) - 2 else "k")
    if PAPER[n] != "no EMD":
        a, b = (PAPER[n][3:].split("-") + [PAPER[n][3:]])[:2]
        ax.add_patch(plt.Rectangle((int(b) - 1.5, int(a) - 1.5), 1, 1, fill=False, ec="red", lw=2.5))
    ax.set_xticks(range(8), range(1, 9)); ax.set_yticks(range(8), range(1, 9))
    ax.set_xlabel("last IMF"); ax.set_ylabel("first IMF"); ax.grid(False)
    ax.set_title(f"{n.upper()}  (no EMD: {mean[n]['no EMD']:.2f} dB)")
    plt.colorbar(im, ax=ax, fraction=0.046, label="SNRimp (dB)")
plt.tight_layout(); plt.show()

## Paper's choice vs best range vs no EMD, per record

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4.2), sharey=False)
for ax, n in zip(axes, PAPER):
    d = imf[imf.noise == n].pivot(index="record", columns="combination", values="SNRimp_dB")
    cols = list(dict.fromkeys([PAPER[n], mean[n].idxmax(), "no EMD"]))
    x = np.arange(len(d))
    for k, c in enumerate(cols):
        ax.bar(x + (k - (len(cols) - 1) / 2) * 0.27, d[c], 0.27, label=c)
    ax.set_xticks(x, d.index, rotation=60, fontsize=8); ax.set_title(n.upper())
    ax.set_ylabel("SNRimp (dB)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Top 8 inputs per noise

In [ ]:
pd.concat({n.upper(): mean[n].sort_values(ascending=False).head(8).round(2).reset_index()
           for n in PAPER}, axis=1)